## 9 Min Aplha

In [ ]:
class TicTacToe:
    def __init__(self):
        self.MAX = 1  # X
        self.MIN = -1 # O

    def getInitialState(self):
        return (0, 0, 0, 0, 0, 0, 0, 0, 0)

    def getPlayer(self, state):
        return self.MAX if state.count(self.MAX) == state.count(self.MIN) else self.MIN

    def getActions(self, state):
        return [i for i, cell in enumerate(state) if cell == 0]

    def getResult(self, state, action):
        new_state = list(state)
        new_state[action] = self.getPlayer(state)
        return tuple(new_state)

    def isTerminal(self, state):
        return self.getUtility(state) != 0 or 0 not in state

    def getUtility(self, state):
        """Standard zero-sum utility: +1 (MAX wins), -1 (MIN wins), 0 (Draw)."""
        win_conditions = [
            (0, 1, 2), (3, 4, 5), (6, 7, 8), # Rows
            (0, 3, 6), (1, 4, 7), (2, 5, 8), # Cols
            (0, 4, 8), (2, 4, 6)             # Diagonals
        ]
        for a, b, c in win_conditions:
            if state[a] == state[b] == state[c] != 0:
                return state[a]
        return 0

    def format_board(self, state):
        """Helper to visualize the 1D tuple as a 2D grid."""
        symbols = {1: 'X', -1: 'O', 0: '.'}
        rows = [state[i:i+3] for i in range(0, 9, 3)]
        return "\n".join([" ".join([symbols[cell] for cell in row]) for row in rows])


game = TicTacToe()
state = game.getInitialState()

print("--- Game Start ---")
print(game.format_board(state))

# Simulate a series of moves: X (4), O (0), X (1), O (8), X (7)
moves = [4, 0, 1, 8, 7]

# T0901 – Formal Game Representation (Tic-Tac-Toe)
# Concepts: Game theory, state-space representation, zero-sum games
for move in moves:
    if not game.isTerminal(state):
        current_player = "MAX (X)" if game.getPlayer(state) == 1 else "MIN (O)"
        print(f"\n{current_player} plays at index {move}:")
        
        state = game.getResult(state, move)
        print(game.format_board(state))
        
        if game.isTerminal(state):
            utility = game.getUtility(state)
            result = "DRAW" if utility == 0 else ("MAX WINS" if utility == 1 else "MIN WINS")
            print(f"\nTERMINAL STATE REACHED: {result} (Utility: {utility})")

In [ ]:
class GameTreeGenerator:
    def __init__(self, game_logic):
        self.game = game_logic
        self.total_nodes = 0
        self.max_depth = 0
        self.visited_states = set()

    def generate(self, state, depth=0):
        self.total_nodes += 1
        self.max_depth = max(self.max_depth, depth)

        
        if self.game.isTerminal(state):
            return

        # Recursive Step: Expand all legal actions
        actions = self.game.getActions(state)
        for action in actions:
            next_state = self.game.getResult(state, action)
            
            if next_state not in self.visited_states:
                self.visited_states.add(next_state)
                self.generate(next_state, depth + 1)

# --- Execution ---
ttt = TicTacToe()
tree_gen = GameTreeGenerator(ttt)

initial_state = ttt.getInitialState()
tree_gen.visited_states.add(initial_state)

print("Generating full game tree... (this may take a moment)")
tree_gen.generate(initial_state)

# T0902 – Game Tree Generation
# Concepts: *Recursively expands all game states from the initial position
# tracking depth and parent-child relationships
print("-" * 30)
print(f"Total Unique States Generated: {tree_gen.total_nodes}")
print(f"Maximum Tree Depth:            {tree_gen.max_depth}")
print("-" * 30)

In [ ]:
import math

class MinimaxAgent:
    def __init__(self, game_logic):
        self.game = game_logic
        self.nodes_explored = 0

    def minimaxDecision(self, state):
        """Returns the best action for the current player at the given state."""
        self.nodes_explored = 0  # Reset counter for each decision
        player = self.game.getPlayer(state)
        best_action = None
        
        actions = self.game.getActions(state)
        
        if player == self.game.MAX:
            v = -math.inf
            for action in actions:
                # Get the value from the perspective of the next player (MIN)
                min_val = self.minValue(self.game.getResult(state, action))
                if min_val > v:
                    v = min_val
                    best_action = action
        else:
            v = math.inf
            for action in actions:
                # Get the value from the perspective of the next player (MAX)
                max_val = self.maxValue(self.game.getResult(state, action))
                if max_val < v:
                    v = max_val
                    best_action = action
                    
        return best_action, v

    def maxValue(self, state):
        self.nodes_explored += 1
        if self.game.isTerminal(state):
            return self.game.getUtility(state)
        
        v = -math.inf
        for action in self.game.getActions(state):
            v = max(v, self.minValue(self.game.getResult(state, action)))
        return v

    def minValue(self, state):
        self.nodes_explored += 1
        if self.game.isTerminal(state):
            return self.game.getUtility(state)
        
        v = math.inf
        for action in self.game.getActions(state):
            v = min(v, self.maxValue(self.game.getResult(state, action)))
        return v

# --- Execution & Demonstration ---
ttt = TicTacToe() # From Task 1
agent = MinimaxAgent(ttt)


test_state = (1, -1, 0, 1, 0, 0, -1, 0, 0)

print("Current Board State:")
print(ttt.format_board(test_state))

best_move, value = agent.minimaxDecision(test_state)

# T0903 – Minimax Algorithm
# Concepts: Adversarial search, optimal play, recursive minimax
print("-" * 30)
print(f"Player to move:   {'MAX (X)' if ttt.getPlayer(test_state) == 1 else 'MIN (O)'}")
print(f"Best Action Index: {best_move}")
print(f"Minimax Value:     {value}")
print(f"Nodes Explored:    {agent.nodes_explored}")
print("-" * 30)

In [ ]:
import math

class TicTacToe:
    """T1: Formal Game Representation"""
    def __init__(self):
        self.MAX = 1  # X
        self.MIN = -1 # O

    def getInitialState(self):
        return (0, 0, 0, 0, 0, 0, 0, 0, 0)

    def getPlayer(self, state):
        return self.MAX if state.count(self.MAX) == state.count(self.MIN) else self.MIN

    def getActions(self, state):
        return [i for i, cell in enumerate(state) if cell == 0]

    def getResult(self, state, action):
        new_state = list(state)
        new_state[action] = self.getPlayer(state)
        return tuple(new_state)

    def isTerminal(self, state):
        return self.getUtility(state) != 0 or 0 not in state

    def getUtility(self, state):
        win_conditions = [
            (0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6),
            (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)
        ]
        for a, b, c in win_conditions:
            if state[a] == state[b] == state[c] != 0:
                return state[a]
        return 0

    def format_board(self, state):
        symbols = {1: 'X', -1: 'O', 0: '.'}
        rows = [state[i:i+3] for i in range(0, 9, 3)]
        return "\n".join([" ".join([symbols[cell] for cell in row]) for row in rows])


def heuristic_evaluation(state):
    """T4: Mandatory Heuristic Evaluation Function"""
    win_conditions = [
        (0, 1, 2), (3, 4, 5), (6, 7, 8), # Rows
        (0, 3, 6), (1, 4, 7), (2, 5, 8), # Cols
        (0, 4, 8), (2, 4, 6)             # Diagonals
    ]
    
    total_score = 0
    for line in win_conditions:
        cells = [state[i] for i in line]
        max_count = cells.count(1)
        min_count = cells.count(-1)
        empty_count = cells.count(0)
        
        # Scoring Rules for MAX (X)
        if max_count == 3: total_score += 100
        elif max_count == 2 and empty_count == 1: total_score += 10
        elif max_count == 1 and empty_count == 2: total_score += 1
        
        # Scoring Rules for MIN (O)
        if min_count == 3: total_score -= 100
        elif min_count == 2 and empty_count == 1: total_score -= 10
        elif min_count == 1 and empty_count == 2: total_score -= 1
            
    return total_score


class DepthLimitedMinimax:
    """T3 & 4: Minimax with Depth Limiting"""
    def __init__(self, game_logic):
        self.game = game_logic

    def minimaxDecision(self, state, depth_limit):
        player = self.game.getPlayer(state)
        best_action = None
        
        actions = self.game.getActions(state)
        
        if player == self.game.MAX:
            v = -math.inf
            for action in actions:
                val = self.minValue(self.game.getResult(state, action), depth_limit - 1, 1)
                if val > v:
                    v = val
                    best_action = action
        else:
            v = math.inf
            for action in actions:
                val = self.maxValue(self.game.getResult(state, action), depth_limit - 1, 1)
                if val < v:
                    v = val
                    best_action = action
        return best_action, v

    def maxValue(self, state, depth, current_depth):
        if self.game.isTerminal(state):
            # Multiply utility by 100 to outweigh any heuristic score
            return self.game.getUtility(state) * 100
        if depth == 0:
            return heuristic_evaluation(state)
        
        v = -math.inf
        for action in self.game.getActions(state):
            v = max(v, self.minValue(self.game.getResult(state, action), depth - 1, current_depth + 1))
        return v

    def minValue(self, state, depth, current_depth):
        if self.game.isTerminal(state):
            return self.game.getUtility(state) * 100
        if depth == 0:
            return heuristic_evaluation(state)
        
        v = math.inf
        for action in self.game.getActions(state):
            v = min(v, self.maxValue(self.game.getResult(state, action), depth - 1, current_depth + 1))
        return v


if __name__ == "__main__":
    ttt = TicTacToe()
    agent = DepthLimitedMinimax(ttt)
    
    test_state = (-1, 0, 0, 0, 1, 0, 0, 0, 0)
    
    print("Initial Test State:")
    print(ttt.format_board(test_state))
    print("-" * 30)

    # Experimental Analysis for Depth 1, 2, and 3
    for d_limit in [1, 2, 3]:
        move, eval_val = agent.minimaxDecision(test_state, d_limit)

        # T0904 – Depth-Limited Minimax with Heuristic Evaluation
        # Concepts: Depth-limited search, heuristic evaluation, trade-off between speed and accuracy
        print(f"RESULTS FOR DEPTH_LIMIT = {d_limit}")
        print(f"Selected Optimal Move: index {move}")
        print(f"Computed Evaluation:   {eval_val}")
        print(f"Max Depth Explored:    {d_limit}")
        print("-" * 30)

In [ ]:
import math

class TicTacToe:
    def __init__(self):
        self.MAX, self.MIN = 1, -1

    def getInitialState(self):
        return (0, 0, 0, 0, 0, 0, 0, 0, 0)

    def getPlayer(self, state):
        return self.MAX if state.count(self.MAX) == state.count(self.MIN) else self.MIN

    def getActions(self, state):
        return [i for i, cell in enumerate(state) if cell == 0]

    def getResult(self, state, action):
        new_state = list(state)
        new_state[action] = self.getPlayer(state)
        return tuple(new_state)

    def isTerminal(self, state):
        return self.getUtility(state) != 0 or 0 not in state

    def getUtility(self, state):
        wins = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
        for a, b, c in wins:
            if state[a] == state[b] == state[c] != 0:
                return state[a]
        return 0

class AdversarialSearch:
    def __init__(self, game):
        self.game = game
        self.nodes_visited = 0
        self.nodes_pruned = 0

    # --- Standard Minimax (Task 3) ---
    def minimaxDecision(self, state):
        self.nodes_visited = 0
        self.nodes_pruned = 0
        player = self.game.getPlayer(state)
        best_val = -math.inf if player == self.game.MAX else math.inf
        best_move = None
        
        for action in self.game.getActions(state):
            res = self.game.getResult(state, action)
            val = self.minValue_std(res) if player == self.game.MAX else self.maxValue_std(res)
            if (player == self.game.MAX and val > best_val) or (player == self.game.MIN and val < best_val):
                best_val, best_move = val, action
        return best_move, best_val, self.nodes_visited

    def maxValue_std(self, state):
        self.nodes_visited += 1
        if self.game.isTerminal(state): return self.game.getUtility(state)
        v = -math.inf
        for action in self.game.getActions(state):
            v = max(v, self.minValue_std(self.game.getResult(state, action)))
        return v

    def minValue_std(self, state):
        self.nodes_visited += 1
        if self.game.isTerminal(state): return self.game.getUtility(state)
        v = math.inf
        for action in self.game.getActions(state):
            v = min(v, self.maxValue_std(self.game.getResult(state, action)))
        return v

    # --- Alpha-Beta Pruning (Task 5) ---
    def alphaBetaDecision(self, state):
        self.nodes_visited = 0
        self.nodes_pruned = 0
        player = self.game.getPlayer(state)
        alpha, beta = -math.inf, math.inf
        best_move = None
        
        if player == self.game.MAX:
            v = -math.inf
            for action in self.game.getActions(state):
                val = self.minValue_ab(self.game.getResult(state, action), alpha, beta)
                if val > v:
                    v, best_move = val, action
                alpha = max(alpha, v)
        else:
            v = math.inf
            for action in self.game.getActions(state):
                val = self.maxValue_ab(self.game.getResult(state, action), alpha, beta)
                if val < v:
                    v, best_move = val, action
                beta = min(beta, v)
        return best_move, v, self.nodes_visited, self.nodes_pruned

    def maxValue_ab(self, state, alpha, beta):
        self.nodes_visited += 1
        if self.game.isTerminal(state): return self.game.getUtility(state)
        v = -math.inf
        actions = self.game.getActions(state)
        for i, action in enumerate(actions):
            v = max(v, self.minValue_ab(self.game.getResult(state, action), alpha, beta))
            if v >= beta:
                self.nodes_pruned += (len(actions) - (i + 1))
                return v
            alpha = max(alpha, v)
        return v

    def minValue_ab(self, state, alpha, beta):
        self.nodes_visited += 1
        if self.game.isTerminal(state): return self.game.getUtility(state)
        v = math.inf
        actions = self.game.getActions(state)
        for i, action in enumerate(actions):
            v = min(v, self.maxValue_ab(self.game.getResult(state, action), alpha, beta))
            if v <= alpha:
                self.nodes_pruned += (len(actions) - (i + 1))
                return v
            beta = min(beta, v)
        return v

# --- Comparative Experiment ---
game = TicTacToe()
search = AdversarialSearch(game)


initial_state = (1, 0, -1, 0, 1, 0, 0, 0, -1)

m_move, m_val, m_nodes = search.minimaxDecision(initial_state)
ab_move, ab_val, ab_nodes, ab_pruned = search.alphaBetaDecision(initial_state)

# T0905 – Alpha-Beta Pruning
# Concepts: Branch-and-bound pruning, alpha/beta bounds, efficiency improvement over Minimax
print(f"{'Algorithm':<12} | {'Nodes Visited':<13} | {'Nodes Pruned':<12} | {'Optimal Move':<12} | {'Utility':<7}")
print("-" * 75)
print(f"{'Minimax':<12} | {m_nodes:<13} | {'0':<12} | {m_move:<12} | {m_val:<7}")
print(f"{'Alpha-Beta':<12} | {ab_nodes:<13} | {ab_pruned:<12} | {ab_move:<12} | {ab_val:<7}")

## 10 CSP

In [ ]:
variables = ['WA', 'NT', 'SA', 'Q', 'NSW', 'V', 'T']
domains   = {v: ['Red', 'Green', 'Blue'] for v in variables}

# Binary inequality constraints (adjacency list)
adjacency = {
    'WA':  ['NT', 'SA'],
    'NT':  ['WA', 'SA', 'Q'],
    'SA':  ['WA', 'NT', 'Q', 'NSW', 'V'],
    'Q':   ['NT', 'SA', 'NSW'],
    'NSW': ['Q', 'SA', 'V'],
    'V':   ['SA', 'NSW'],
    'T':   []
}

# Constraints as (scope, relation) pairs
constraints = [(u, v) for u, neighbors in adjacency.items() for v in neighbors if u < v]

# T1001 – Formal CSP Modeling: Map Coloring Problem 
# Concepts: CSP formulation, adjacency constraints, constraint graph representation
print("=== CSP: Australia Map Coloring ===")
print(f"Variables : {variables}")
print(f"Domain    : {list(domains['WA'])}")
print(f"Constraints (binary inequality: Xi ≠ Xj):")
for u, v in constraints:
    print(f"  {u} ≠ {v}")
print(f"Total constraints: {len(constraints)}")

In [ ]:
N = 8
def is_safe(assignment, row, col):
    for r, c in enumerate(assignment):
        if c == col or abs(r - row) == abs(c - col):
            return False
    return True

def solve_nqueens(assignment=[]):
    if len(assignment) == N:
        return assignment
    row = len(assignment)
    for col in range(N):
        if is_safe(assignment, row, col):
            result = solve_nqueens(assignment + [col])
            if result:
                return result
    return None

solution = solve_nqueens()
print(f"N-Queens (N={N}) Solution:")
print(f"Column positions per row: {solution}")
board = [['.' for _ in range(N)] for _ in range(N)]

# T1002 – Backtracking Search: N-Queens (N=8)
# Concepts: Backtracking, constraint checking, combinatorial search
for row, col in enumerate(solution):
    board[row][col] = 'Q'
for row in board:
    print(' '.join(row))

In [ ]:
trace_log = []

def solve_nqueens_trace(assignment=[]):
    if len(assignment) == N:
        return assignment
    row = len(assignment)
    for col in range(N):
        trace_log.append(f"Trying   Row={row}, Col={col} | Partial: {assignment}")
        if is_safe(assignment, row, col):
            result = solve_nqueens_trace(assignment + [col])
            if result:
                return result
        else:
            trace_log.append(f"  CONFLICT at Row={row}, Col={col} → Backtrack")
    trace_log.append(f"  BACKTRACK from Row={row}")
    return None

trace_log = []
sol = solve_nqueens_trace()
# Print first 40 lines to keep output readable
for line in trace_log[:40]:
    print(line)

# T1003 – Backtracking Execution Trace(N-Queen Tracking)
# Concepts: records each attempt, constraint violation, and backtrack event to show the depth-first search process.
print(f"... ({len(trace_log)} total trace events)")
print(f"Solution: {sol}")

In [ ]:
import copy

def forward_checking(variables, domains, adjacency, assignment={}):
    if len(assignment) == len(variables):
        return assignment

    # Pick next unassigned variable
    unassigned = [v for v in variables if v not in assignment]
    var = unassigned[0]

    for value in domains[var]:
        new_domains = copy.deepcopy(domains)
        new_domains[var] = [value]
        consistent = True

        # Prune neighbours
        for neighbor in adjacency.get(var, []):
            if neighbor not in assignment:
                new_domains[neighbor] = [v for v in new_domains[neighbor] if v != value]
                if not new_domains[neighbor]:
                    print(f"  Domain wipeout for {neighbor} after {var}={value}")
                    consistent = False
                    break
                else:
                    print(f"  After {var}={value}: D({neighbor})={new_domains[neighbor]}")

        if consistent:
            result = forward_checking(variables, new_domains, adjacency, {**assignment, var: value})
            if result:
                return result
    return None

# T1004 – Forward Checking: Map Coloring CSP
# Concepts: Look-ahead search, domain pruning, early failure detection
result = forward_checking(variables, {v: ['Red','Green','Blue'] for v in variables}, adjacency)
print("=== Valid Coloring ===")
for k, v in result.items():
    print(f"  {k} = {v}")

In [ ]:
N4 = 4
stats = {"recursive_calls": 0, "constraint_checks": 0, "backtracks": 0}

def bt_4queens(assignment, s):
    s["recursive_calls"] += 1
    if len(assignment) == N4:
        return assignment
    row = len(assignment)
    for col in range(N4):
        s["constraint_checks"] += 1
        if is_safe(assignment, row, col):
            result = bt_4queens(assignment + [col], s)
            if result: return result
    s["backtracks"] += 1
    return None

def bt_fc_4queens(assignment, domains, s):
    s["recursive_calls"] += 1
    if len(assignment) == N4:
        return assignment
    row = len(assignment)
    for col in domains[row]:
        s["constraint_checks"] += 1
        if is_safe(assignment, row, col):
            new_domains = [d[:] for d in domains]
            valid = True
            for future_row in range(row + 1, N4):
                new_domains[future_row] = [c for c in new_domains[future_row]
                                           if is_safe(assignment + [col], future_row, c)]
                if not new_domains[future_row]:
                    valid = False; break
            if valid:
                result = bt_fc_4queens(assignment + [col], new_domains, s)
                if result: return result
    s["backtracks"] += 1
    return None

s1 = {"recursive_calls": 0, "constraint_checks": 0, "backtracks": 0}
bt_4queens([], s1)

s2 = {"recursive_calls": 0, "constraint_checks": 0, "backtracks": 0}
bt_fc_4queens([], [list(range(N4)) for _ in range(N4)], s2)

# T1005 – Comparative Evaluation: Backtracking vs Backtracking + Forward Checking
# Concepts: Performance benchmarking, search efficiency, CSP strategies
# Forward Checking on the 4-Queens problem.
print(f"{'Strategy':<25} {'Recursive Calls':>16} {'Constraint Checks':>18} {'Backtracks':>11}")
print("-" * 75)
print(f"{'Backtracking':<25} {s1['recursive_calls']:>16} {s1['constraint_checks']:>18} {s1['backtracks']:>11}")
print(f"{'Backtracking + FC':<25} {s2['recursive_calls']:>16} {s2['constraint_checks']:>18} {s2['backtracks']:>11}")

In [ ]:
courses      = ['C1', 'C2', 'C3', 'C4', 'C5']
time_slots   = ['T1', 'T2', 'T3', 'T4', 'T5']

# Instructor conflicts: pairs that share the same instructor
instructor_conflicts = [('C1', 'C3'), ('C2', 'C4')]

# Student-group overlaps: pairs that share students
student_conflicts    = [('C1', 'C2'), ('C3', 'C5')]

# Unary constraint: C4 must be in T1 or T2 only
unary_domain = {'C4': ['T1', 'T2']}

domains_sched = {c: unary_domain.get(c, time_slots[:]) for c in courses}

def schedule_ok(var, val, assignment):
    for assigned_var, assigned_val in assignment.items():
        if assigned_val == val:
            pair = tuple(sorted([var, assigned_var]))
            if pair in [tuple(sorted(p)) for p in instructor_conflicts + student_conflicts]:
                return False
    return True

def solve_schedule(idx, assignment):
    if idx == len(courses): return assignment
    var = courses[idx]
    for val in domains_sched[var]:
        if schedule_ok(var, val, assignment):
            result = solve_schedule(idx + 1, {**assignment, var: val})
            if result: return result
    return None

sched = solve_schedule(0, {})

# T1006 – University Course Scheduling using CSP
# Concepts: Real-world CSP modelling, constraint encoding, backtracking solver
print("=== University Course Schedule ===")
if sched:
    for c, t in sched.items():
        print(f"  {c} → {t}")
    print("Constraint verification:")
    all_ok = True
    for c1, c2 in instructor_conflicts + student_conflicts:
        if sched[c1] == sched[c2]:
            print(f"  CONFLICT: {c1} and {c2} both in {sched[c1]}"); all_ok = False
    if all_ok: print("  All constraints satisfied ✓")

In [ ]:
meetings = {
    'M1': {'time': (9, 10),  'attendees': 10, 'needs_projector': True},
    'M2': {'time': (9, 11),  'attendees': 5,  'needs_projector': False},
    'M3': {'time': (10, 12), 'attendees': 20, 'needs_projector': False},
    'M4': {'time': (11, 13), 'attendees': 8,  'needs_projector': True},
}
rooms = {
    'R1': {'capacity': 15, 'has_projector': True},
    'R2': {'capacity': 25, 'has_projector': False},
    'R3': {'capacity': 10, 'has_projector': True},
}

def overlaps(m1, m2):
    s1, e1 = meetings[m1]['time']; s2, e2 = meetings[m2]['time']
    return not (e1 <= s2 or e2 <= s1)

def room_ok(meeting, room):
    m = meetings[meeting]; r = rooms[room]
    if m['attendees'] > r['capacity']: return False
    if m['needs_projector'] and not r['has_projector']: return False
    return True

import copy

def allocate(meeting_list, domains, assignment={}):
    if not meeting_list: return assignment
    m = meeting_list[0]
    for r in domains[m]:
        if not room_ok(m, r): continue
        conflict = any(overlaps(m, am) and ar == r for am, ar in assignment.items())
        if conflict: continue
        new_domains = copy.deepcopy(domains)
        consistent = True
        for future_m in meeting_list[1:]:
            if overlaps(m, future_m) and r in new_domains[future_m]:
                new_domains[future_m].remove(r)
                print(f"  After {m}→{r}: removed {r} from domain of {future_m}")
                if not new_domains[future_m]:
                    consistent = False; break
        if consistent:
            result = allocate(meeting_list[1:], new_domains, {**assignment, m: r})
            if result: return result
    return None

# T1007 – Meeting Room Allocation with Forward Checking
# Concepts: Resource allocation CSP, capacity/equipment constraints, dynamic domain pruning
init_domains = {m: list(rooms.keys()) for m in meetings}
allocation = allocate(list(meetings.keys()), init_domains)
print("=== Room Allocation ===")
for m, r in allocation.items():
    print(f"  {m} → {r}")

In [ ]:
# Variables: (row, col) for each of the 81 cells
sudoku_variables = [(r, c) for r in range(9) for c in range(9)]

# Domain: each cell can hold digits 1-9
sudoku_domains = {v: list(range(1, 10)) for v in sudoku_variables}

# Constraints: all-different within rows, columns, and 3x3 subgrids
sudoku_constraints = []

# Row constraints
for r in range(9):
    row_cells = [(r, c) for c in range(9)]
    for i in range(len(row_cells)):
        for j in range(i + 1, len(row_cells)):
            sudoku_constraints.append((row_cells[i], row_cells[j]))

# Column constraints
for c in range(9):
    col_cells = [(r, c) for r in range(9)]
    for i in range(len(col_cells)):
        for j in range(i + 1, len(col_cells)):
            sudoku_constraints.append((col_cells[i], col_cells[j]))

# 3x3 subgrid constraints
for br in range(3):
    for bc in range(3):
        box = [(br*3+r, bc*3+c) for r in range(3) for c in range(3)]
        for i in range(len(box)):
            for j in range(i + 1, len(box)):
                sudoku_constraints.append((box[i], box[j]))

print("=== Sudoku CSP Definition ===")
print(f"Variables  : {len(sudoku_variables)} cells (row, col)")
print(f"Domain     : {{1, 2, 3, 4, 5, 6, 7, 8, 9}} per cell")
print(f"Constraints: {len(sudoku_constraints)} all-different pairs")
print(f"Constraint types:")

# T1008 – CSP Formulation for Sudoku (9×9)
# Concepts: Constraint modelling, all-different constraints, subgrid indexing
print(f"  Row constraints    : 9 rows × C(9,2) = {9 * 36} pairs")
print(f"  Column constraints : 9 cols × C(9,2) = {9 * 36} pairs")
print(f"  Subgrid constraints: 9 boxes × C(9,2) = {9 * 36} pairs")
print(f"Sample constraints: {sudoku_constraints[:3]}")
print(f"All constraints: Xi ≠ Xj for every pair (Xi, Xj) sharing a row/col/box")


## 11 CSP (LSV)

In [ ]:
N_mrv = 4

def get_mrv_var(domains, assignment):
    """Select unassigned variable with smallest domain (MRV)."""
    unassigned = [v for v in range(N_mrv) if v not in assignment]
    return min(unassigned, key=lambda v: len(domains[v]))

def is_safe_mrv(assignment, row, col):
    for r, c in assignment.items():
        if c == col or abs(r - row) == abs(c - col):
            return False
    return True

def update_domains(domains, assignment, row, col):
    """Remove values made inconsistent by assigning col to row."""
    new_domains = [d[:] for d in domains]
    for future_row in range(N_mrv):
        if future_row not in assignment:
            new_domains[future_row] = [
                c for c in new_domains[future_row]
                if not (c == col or abs(future_row - row) == abs(c - col))
            ]
    return new_domains

def solve_mrv(assignment, domains, order_log):
    if len(assignment) == N_mrv:
        return assignment
    var = get_mrv_var(domains, assignment)
    order_log.append(f"MRV selected: Row {var} (domain size={len(domains[var])}, values={domains[var]})")
    for val in domains[var]:
        if is_safe_mrv(assignment, var, val):
            new_domains = update_domains(domains, assignment, var, val)
            result = solve_mrv({**assignment, var: val}, new_domains, order_log)
            if result: return result
    return None

# T1101 – MRV Heuristic: 4-Queens Problem
# Concepts: MRV variable ordering, dynamic domain tracking, heuristic-guided search
init_domains = [list(range(N_mrv)) for _ in range(N_mrv)]
order_log = []
sol = solve_mrv({}, init_domains, order_log)
print("=== MRV-Based 4-Queens ===")
for entry in order_log:
    print(" ", entry)
print(f"Solution (row → col): {[sol[r] for r in range(N_mrv)]}")

In [ ]:
adjacency_lcv = {
    'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'],
    'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'],
    'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW']
}
variables_lcv = ['WA', 'NT', 'SA', 'Q', 'NSW', 'V']
domains_lcv   = {v: ['Red', 'Green', 'Blue'] for v in variables_lcv}

def count_eliminated(var, val, domains, assignment):
    """Count how many values are eliminated from unassigned neighbours."""
    total = 0
    for nb in adjacency_lcv.get(var, []):
        if nb not in assignment:
            total += sum(1 for d in domains[nb] if d == val)
    return total

def lcv_order(var, domains, assignment):
    """Sort domain values by how few they eliminate (LCV)."""
    return sorted(domains[var], key=lambda val: count_eliminated(var, val, domains, assignment))

def solve_lcv(variables, domains, assignment={}):
    if len(assignment) == len(variables): return assignment
    var = next(v for v in variables if v not in assignment)
    ordered_vals = lcv_order(var, domains, assignment)
    print(f"Variable Selected: {var}")
    print(f"LCV Order: {ordered_vals}")
    for val in ordered_vals:
        if all(assignment.get(nb) != val for nb in adjacency_lcv.get(var, [])):
            result = solve_lcv(variables, domains, {**assignment, var: val})
            if result: return result
    return None

# T1102 – LCV Heuristic: Map Coloring CSP
# Concepts: LCV value ordering, constraint counting, backtracking integration
solution_lcv = solve_lcv(variables_lcv, domains_lcv)
print("=== Final Assignment ===")
for k, v in solution_lcv.items():
    print(f"  {k} = {v}")

In [ ]:
from collections import deque

def revise(domains, xi, xj, constraint):
    """Remove values from D(Xi) with no supporting value in D(Xj). Returns True if revised."""
    removed = []
    for x in domains[xi][:]:
        if not any(constraint(x, y) for y in domains[xj]):
            domains[xi].remove(x)
            removed.append(x)
    if removed:
        print(f"Revise({xi}, {xj}): Removed {removed}")
        return True
    return False

def ac3(variables, domains, constraints_arcs):
    queue = deque(constraints_arcs)
    while queue:
        xi, xj, cond = queue.popleft()
        if revise(domains, xi, xj, cond):
            if not domains[xi]:
                print(f"Domain of {xi} is empty — CSP has no solution!")
                return False
            for xk, _, c in constraints_arcs:
                if xk != xj and _ == xi:
                    queue.append((xk, xi, c))
    return True

domains_ac3 = {'X1': [1,2,3,4], 'X2': [1,2,3,4], 'X3': [1,2,3,4]}
arcs = [
    ('X1', 'X2', lambda x, y: x < y),
    ('X2', 'X1', lambda x, y: x > y),
    ('X2', 'X3', lambda x, y: x < y),
    ('X3', 'X2', lambda x, y: x > y),
]

# T1103 – AC-3 Algorithm: Binary CSP (X1 < X2 < X3)
# Concepts: Arc consistency, constraint propagation, domain reduction
print("=== AC-3 on X1 < X2 < X3 ===")
ac3(['X1','X2','X3'], domains_ac3, arcs)
print(f"Final Domains:")
for var, dom in domains_ac3.items():
    print(f"  D({var}) = {dom}")

In [ ]:
from collections import deque
import copy

grid = [
    [1, 0, 0, 4],
    [0, 0, 3, 0],
    [0, 3, 0, 0],
    [2, 0, 0, 1],
]
N_s = 4

# Initialize domains
domains_s = {}
for r in range(N_s):
    for c in range(N_s):
        if grid[r][c] != 0:
            domains_s[(r,c)] = [grid[r][c]]
        else:
            domains_s[(r,c)] = list(range(1, N_s+1))

# Build all-different constraint pairs
def get_peers(r, c):
    peers = set()
    for i in range(N_s): peers.add((r, i)); peers.add((i, c))
    br, bc = (r//2)*2, (c//2)*2
    for dr in range(2):
        for dc in range(2):
            peers.add((br+dr, bc+dc))
    peers.discard((r, c))
    return peers

def revise_sudoku(domains, xi, xj):
    revised = False
    for val in domains[xi][:]:
        if domains[xj] == [val]:          # xj is fixed to val → remove from xi
            domains[xi].remove(val)
            print(f"  Revise({xi},{xj}): Removed {val}")
            revised = True
    return revised

def ac3_sudoku(domains):
    queue = deque()
    for cell in domains:
        for peer in get_peers(*cell):
            queue.append((cell, peer))
    while queue:
        xi, xj = queue.popleft()
        if revise_sudoku(domains, xi, xj):
            if not domains[xi]: return False
            for xk in get_peers(*xi):
                if xk != xj: queue.append((xk, xi))
    return True

# T1104 – AC-3 on 4×4 Sudoku
# Concepts: AC-3 applied to real puzzle, all-different constraints, domain reduction without backtracking
print("=== AC-3 on 4×4 Sudoku ===")
ac3_sudoku(domains_s)
print("Reduced Domains for Empty Cells:")
for r in range(N_s):
    for c in range(N_s):
        if grid[r][c] == 0:
            print(f"  Cell ({r},{c}) → {domains_s[(r,c)]}")


In [ ]:
print("=== CSP Formulation: SEND + MORE = MONEY ===")
print("Variables : S, E, N, D, M, O, R, Y, C1, C2, C3, C4  (C = carry bits)")
print("Domains   : S,E,N,D,O,R,Y ∈ {0..9},  M ∈ {1..9},  C1..C4 ∈ {0,1}")
print()
print("Constraints:")
print("  1. All-different: S,E,N,D,M,O,R,Y are all distinct digits")
print("  2. Leading digits: S ≠ 0, M ≠ 0")
print()
print("  Column-wise arithmetic (right to left):")
print("  Col 0 (units)    : D + E         = Y + 10*C1")
print("  Col 1 (tens)     : N + R + C1    = E + 10*C2")
print("  Col 2 (hundreds) : E + O + C2    = N + 10*C3")
print("  Col 3 (thousands): S + M + C3    = O + 10*C4")
print("  Col 4 (carry)    : C4            = M")
print()

# Programmatic structure
csp_structure = {
    "variables": ['S','E','N','D','M','O','R','Y','C1','C2','C3','C4'],
    "domains": {
        'S':list(range(1,10)), 'M':list(range(1,10)),
        **{v:list(range(10)) for v in ['E','N','D','O','R','Y']},
        **{f'C{i}':[0,1] for i in range(1,5)}
    },
    "constraints": [
        "D + E == Y + 10*C1",
        "N + R + C1 == E + 10*C2",
        "E + O + C2 == N + 10*C3",
        "S + M + C3 == O + 10*C4",
        "C4 == M",
        "AllDifferent(S,E,N,D,M,O,R,Y)"
    ]
}

# T1105 – Cryptarithmetic CSP Modeling: SEND + MORE = MONEY
# Concepts: CSP formulation, carry propagation, all-different constraint, arithmetic encoding
print("Programmatic CSP structure:")
for k, v in csp_structure.items():
    if k == "domains":
        print(f"  {k}: (shown per variable above)")
    else:
        print(f"  {k}: {v}")


In [ ]:
from itertools import permutations

def solve_two_plus_two():
    letters = ['T','W','O','F','U','R']
    for perm in permutations(range(10), len(letters)):
        assignment = dict(zip(letters, perm))
        T,W,O,F,U,R = [assignment[l] for l in letters]
        if T == 0 or F == 0: continue      # leading digit constraint
        TWO  = 100*T + 10*W + O
        FOUR = 1000*F + 100*O + 10*U + R
        if TWO + TWO == FOUR:
            return assignment
    return None

# T1106 – Cryptarithmetic Solver: TWO + TWO = FOUR (Backtracking + MRV + LCV)
# Concepts: Constraint propagation in arithmetic puzzles, heuristic-guided search
sol = solve_two_plus_two()
print("=== TWO + TWO = FOUR ===")
if sol:
    T,W,O,F,U,R = sol['T'],sol['W'],sol['O'],sol['F'],sol['U'],sol['R']
    TWO  = 100*T + 10*W + O
    FOUR = 1000*F + 100*O + 10*U + R
    print(f"  T={T}, W={W}, O={O}, F={F}, U={U}, R={R}")
    print(f"  Verification: {TWO} + {TWO} = {FOUR}  →  {'✓' if TWO+TWO==FOUR else '✗'}")

In [ ]:
exams = ['E1','E2','E3','E4','E5']
slots = ['Morning','Afternoon','Evening']
room_capacity = {'Morning': 2, 'Afternoon': 2, 'Evening': 2}

# Shared students: these pairs cannot share a time slot
student_conflicts_exams = [('E1','E2'),('E2','E3'),('E3','E4'),('E4','E5')]

# Priority: E1 and E2 must be scheduled before E5 (handled via variable ordering)
exam_domains = {e: slots[:] for e in exams}

def count_remaining(exam, val, domains, assignment):
    count = 0
    for neighbor in [b for a,b in student_conflicts_exams if a==exam] +                     [a for a,b in student_conflicts_exams if b==exam]:
        if neighbor not in assignment and val in domains[neighbor]:
            count += 1
    return count

def mrv_exam(domains, assignment):
    unassigned = [e for e in exams if e not in assignment]
    return min(unassigned, key=lambda e: len(domains[e]))

def lcv_exam(exam, domains, assignment):
    return sorted(domains[exam], key=lambda v: count_remaining(exam, v, domains, assignment))

import copy

def solve_exams(domains, assignment={}, log=[]):
    if len(assignment) == len(exams): return assignment
    var = mrv_exam(domains, assignment)
    ordered = lcv_exam(var, domains, assignment)
    log.append(f"MRV selected: {var} | LCV order: {ordered}")
    for val in ordered:
        slot_count = sum(1 for v in assignment.values() if v == val)
        if slot_count >= room_capacity[val]: continue
        conflict = any(assignment.get(b)==val for a,b in student_conflicts_exams if a==var) or                    any(assignment.get(a)==val for a,b in student_conflicts_exams if b==var)
        if not conflict:
            new_domains = copy.deepcopy(domains)
            for nb in [b for a,b in student_conflicts_exams if a==var] +                       [a for a,b in student_conflicts_exams if b==var]:
                if nb not in assignment and val in new_domains[nb]:
                    new_domains[nb].remove(val)
            result = solve_exams(new_domains, {**assignment, var: val}, log)
            if result: return result
    return None

# T1107 – Exam Timetabling CSP with MRV + LCV
# Concepts: Real-world scheduling CSP, MRV + LCV combined, conflict detection
log_entries = []
schedule = solve_exams(exam_domains, log=log_entries)
print("=== Exam Schedule ===")
for entry in log_entries: print(" ", entry)
print()
if schedule:
    for e, s in schedule.items(): print(f"  {e} → {s}")

In [ ]:
from collections import deque

towers    = ['T1','T2','T3','T4','T5']
all_freqs = ['F1','F2','F3','F4']

# Adjacent towers (must differ in frequency)
tower_adjacency = {
    'T1': ['T2','T3'], 'T2': ['T1','T4'],
    'T3': ['T1','T4'], 'T4': ['T2','T3','T5'], 'T5': ['T4']
}

# Hardware limitations (unary restrictions)
hw_restrictions = {'T1': ['F1'], 'T3': ['F3']}  # these frequencies NOT available

tower_domains = {t: [f for f in all_freqs if f not in hw_restrictions.get(t, [])]
                 for t in towers}

def revise_freq(domains, ti, tj):
    revised = False
    for f in domains[ti][:]:
        if all(f == g for g in domains[tj]):   # every value in Tj conflicts
            domains[ti].remove(f)
            print(f"  Arc ({ti},{tj}): Removed {f} from {ti}")
            revised = True
    return revised

def ac3_freq(domains, adjacency):
    queue = deque()
    for t, neighbors in adjacency.items():
        for nb in neighbors:
            queue.append((t, nb))
    while queue:
        ti, tj = queue.popleft()
        if revise_freq(domains, ti, tj):
            if not domains[ti]:
                print(f"  Domain of {ti} wiped out!"); return False
            for tk in adjacency[ti]:
                if tk != tj: queue.append((tk, ti))
    return True

print("=== AC-3: Frequency Assignment ===")
print("Initial domains:", {t: d for t,d in tower_domains.items()})
print()

# T1108 – Network Frequency Assignment using AC-3
# Concepts: AC-3 preprocessing, wireless network constraint modelling, domain reduction
ac3_freq(tower_domains, tower_adjacency)
print("Reduced Domains:")
for t, d in tower_domains.items():
    print(f"  {t} → {d}")


## 12 Regres

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- Synthetic Salary Data (mimics YearsExperience vs Salary) ---
np.random.seed(42)
X_raw = np.linspace(1, 10, 30)
y     = 30000 + 9000 * X_raw + np.random.randn(30) * 3000

# Normalize feature
scaler_1201 = StandardScaler()
X_norm = scaler_1201.fit_transform(X_raw.reshape(-1,1)).flatten()

m = len(X_norm)
theta0, theta1 = 0.0, 0.0
alpha, epochs  = 0.1, 500
cost_history   = []

for _ in range(epochs):
    h     = theta0 + theta1 * X_norm
    error = h - y
    J     = (1 / (2 * m)) * np.sum(error ** 2)
    cost_history.append(J)
    theta0 -= alpha * (1/m) * np.sum(error)
    theta1 -= alpha * (1/m) * np.sum(error * X_norm)

print(f"Learned θ0 = {theta0:.2f},  θ1 = {theta1:.2f}")

# T1201 – Univariate Linear Regression from Scratch using Batch Gradient Descent
# Concepts: Gradient descent, cost function (MSE), parameter update, convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_raw, y, color='steelblue', label='Data')
axes[0].plot(X_raw, theta0 + theta1 * X_norm, color='red', label='Regression Line')
axes[0].set_title('Linear Regression (Scratch)'); axes[0].legend()
axes[1].plot(cost_history, color='green')
axes[1].set_title('Cost vs Iterations'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Cost J')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X_sk = X_norm.reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X_sk, y, test_size=0.2, random_state=42)

model_sk = LinearRegression().fit(X_tr, y_tr)
y_pred   = model_sk.predict(X_te)

print(f"sklearn  θ0 = {model_sk.intercept_:.2f},  θ1 = {model_sk.coef_[0]:.2f}")
print(f"Scratch  θ0 = {theta0:.2f},  θ1 = {theta1:.2f}")
print(f"MSE  : {mean_squared_error(y_te, y_pred):.2f}")
print(f"R²   : {r2_score(y_te, y_pred):.4f}")

# T1202 – Univariate Linear Regression using Scikit-learn
# Concepts: sklearn pipeline, train-test split, MSE, R² score, comparison with scratch
plt.figure(figsize=(7, 4))
plt.scatter(X_te, y_te, color='steelblue', label='Test Data')
plt.plot(X_te, y_pred, color='red', label='sklearn')
plt.plot(X_sk, theta0 + theta1 * X_sk, color='orange', linestyle='--', label='Scratch')
plt.title('Sklearn vs Scratch Regression'); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

data = fetch_california_housing()
X_cal, y_cal = pd.DataFrame(data.data, columns=data.feature_names), data.target

scaler_cal = StandardScaler()
X_scaled = scaler_cal.fit_transform(X_cal)

X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_cal, test_size=0.2, random_state=42)
lr_cal = LinearRegression().fit(X_tr, y_tr)
y_pred_cal = lr_cal.predict(X_te)

print(f"MSE : {mean_squared_error(y_te, y_pred_cal):.4f}")
print(f"R²  : {r2_score(y_te, y_pred_cal):.4f}")

coef_df = pd.DataFrame({'Feature': data.feature_names, 'Coefficient': lr_cal.coef_}).sort_values('Coefficient')
print("Feature Coefficients:", coef_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(coef_df['Feature'], coef_df['Coefficient'], color='teal')
axes[0].set_title('Feature Importance (Coefficients)'); axes[0].axvline(0, color='black')

# T1203 – Multivariate Linear Regression: California Housing
# Concepts: Multiple regression, feature scaling, coefficient interpretation, heatmap analysis
df_full = X_cal.copy(); df_full['target'] = y_cal
sns.heatmap(df_full.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1])
axes[1].set_title('Feature Correlation Heatmap')
plt.tight_layout(); plt.show()


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

data_bc = load_breast_cancer()
X_bc, y_bc = data_bc.data, data_bc.target   # Malignant=0, Benign=1

scaler_bc = StandardScaler()
X_bc_s = scaler_bc.fit_transform(X_bc)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc_s, y_bc, test_size=0.2, random_state=42)

# Add bias column
X_tr_b = np.hstack([np.ones((len(X_tr),1)), X_tr])
X_te_b = np.hstack([np.ones((len(X_te),1)), X_te])

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def compute_cost(X, y, theta):
    h = sigmoid(X @ theta)
    h = np.clip(h, 1e-9, 1 - 1e-9)
    return -np.mean(y * np.log(h) + (1 - y) * np.log(1 - h))

theta = np.zeros(X_tr_b.shape[1])
alpha_lr, epochs_lr = 0.1, 300
cost_hist_lr = []

for _ in range(epochs_lr):
    h = sigmoid(X_tr_b @ theta)
    theta -= alpha_lr * (1/len(X_tr_b)) * (X_tr_b.T @ (h - y_tr))
    cost_hist_lr.append(compute_cost(X_tr_b, y_tr, theta))

y_pred_bc = (sigmoid(X_te_b @ theta) >= 0.5).astype(int)

print(f"Accuracy : {accuracy_score(y_te, y_pred_bc):.4f}")
print(f"Precision: {precision_score(y_te, y_pred_bc):.4f}")
print(f"Recall   : {recall_score(y_te, y_pred_bc):.4f}")
print(f"F1-Score : {f1_score(y_te, y_pred_bc):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cost_hist_lr, color='crimson')
axes[0].set_title('Cost vs Iterations'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cost')

# T1204 – Logistic Regression from Scratch using Batch Gradient Descent
# Concepts: Sigmoid activation, binary cross-entropy, gradient descent, classification metrics
cm = confusion_matrix(y_te, y_pred_bc)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Malignant','Benign'], yticklabels=['Malignant','Benign'])
axes[1].set_title('Confusion Matrix')
plt.tight_layout(); plt.show()


## 13 CLass & K-nea

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from matplotlib.colors import ListedColormap

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

scaler_iris = StandardScaler()
X_scaled_iris = scaler_iris.fit_transform(X_iris)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled_iris, y_iris,
                                            test_size=0.2, stratify=y_iris, random_state=42)

# K sweep
k_range = range(1, 21)
k_accuracies = []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr, y_tr)
    k_accuracies.append(accuracy_score(y_te, knn.predict(X_te)))

best_k = k_range.start + k_accuracies.index(max(k_accuracies))
print(f"Best K = {best_k}  (accuracy = {max(k_accuracies):.4f})")

best_knn = KNeighborsClassifier(n_neighbors=best_k).fit(X_tr, y_tr)
y_pred_iris = best_knn.predict(X_te)
print(classification_report(y_te, y_pred_iris, target_names=iris.target_names))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# K vs Accuracy
axes[0].plot(list(k_range), k_accuracies, marker='o', color='steelblue')
axes[0].axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
axes[0].set_title('K vs Accuracy'); axes[0].set_xlabel('K'); axes[0].legend()

# Confusion matrix
cm_iris = confusion_matrix(y_te, y_pred_iris)
sns.heatmap(cm_iris, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=iris.target_names, yticklabels=iris.target_names)
axes[1].set_title('Confusion Matrix')

# T1301 – KNN Classification: Iris Dataset + Hyperparameter Tuning
# Concepts: Distance-based classification, K selection, decision boundaries, precision/recall
# Decision boundary (2 features)
knn2 = KNeighborsClassifier(n_neighbors=best_k).fit(X_tr[:, :2], y_tr)
x_min, x_max = X_tr[:,0].min()-1, X_tr[:,0].max()+1
y_min, y_max = X_tr[:,1].min()-1, X_tr[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min,x_max,200), np.linspace(y_min,y_max,200))
Z = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
colors = ListedColormap(['#FFAAAA','#AAFFAA','#AAAAFF'])
axes[2].contourf(xx, yy, Z, alpha=0.4, cmap=colors)
scatter = axes[2].scatter(X_tr[:,0], X_tr[:,1], c=y_tr, cmap=colors, edgecolors='k', s=30)
axes[2].set_title('Decision Boundary (2 features)')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_curve, roc_auc_score)
from sklearn.preprocessing import StandardScaler

bc = load_breast_cancer()
X_bc2, y_bc2 = bc.data, bc.target
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_bc2, y_bc2, test_size=0.2,
                                                stratify=y_bc2, random_state=42)

results = {}
for criterion in ['gini', 'entropy']:
    dt = DecisionTreeClassifier(criterion=criterion, max_depth=5, random_state=42)
    dt.fit(X_tr2, y_tr2)
    acc = accuracy_score(y_te2, dt.predict(X_te2))
    results[criterion] = (dt, acc)
    print(f"{criterion.capitalize()} accuracy: {acc:.4f}")

best_dt, _ = results['gini']

# Depth vs Accuracy
depths, train_acc, test_acc = [], [], []
for d in range(1, 21):
    dt_d = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr2, y_tr2)
    depths.append(d)
    train_acc.append(accuracy_score(y_tr2, dt_d.predict(X_tr2)))
    test_acc.append(accuracy_score(y_te2, dt_d.predict(X_te2)))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Tree plot
plot_tree(best_dt, feature_names=bc.feature_names, class_names=bc.target_names,
          filled=True, max_depth=3, ax=axes[0][0])
axes[0][0].set_title('Decision Tree (Gini, depth≤3)')

# Feature importance
fi = pd.Series(best_dt.feature_importances_, index=bc.feature_names).nlargest(10)
fi.plot.barh(ax=axes[0][1], color='teal'); axes[0][1].set_title('Top 10 Feature Importances')

# ROC curve
y_prob = best_dt.predict_proba(X_te2)[:,1]
fpr, tpr, _ = roc_curve(y_te2, y_prob)
auc = roc_auc_score(y_te2, y_prob)
axes[1][0].plot(fpr, tpr, color='darkorange', label=f'AUC={auc:.3f}')
axes[1][0].plot([0,1],[0,1],'k--'); axes[1][0].set_title('ROC Curve'); axes[1][0].legend()

# T1302 – Decision Tree Classification: Breast Cancer (Gini vs Entropy + Pruning)
# Concepts: Information gain, Gini impurity, overfitting, cost-complexity pruning, ROC curve

# Depth vs Accuracy
axes[1][1].plot(depths, train_acc, label='Train', marker='o')
axes[1][1].plot(depths, test_acc,  label='Test',  marker='s')
axes[1][1].set_title('Depth vs Accuracy'); axes[1][1].legend()
axes[1][1].set_xlabel('Max Depth')

plt.tight_layout(); plt.show()

In [ ]:
from sklearn.datasets import load_digits
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns

digits = load_digits()
X_dig = digits.data / 16.0   # normalize pixel values
y_dig = digits.target

X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_dig, y_dig, test_size=0.2, random_state=42)
knn_dig = KNeighborsClassifier(n_neighbors=3).fit(X_tr_d, y_tr_d)
y_pred_d = knn_dig.predict(X_te_d)
print(f"Digits KNN Accuracy: {accuracy_score(y_te_d, y_pred_d):.4f}")

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes[0]):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f"Label: {digits.target[i]}"); ax.axis('off')

cm_d = confusion_matrix(y_te_d, y_pred_d)
sns.heatmap(cm_d, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1][0])
axes[1][0].set_title('Confusion Matrix')

# Misclassified samples
wrong_idx = np.where(y_pred_d != y_te_d)[0]
for j, ax in enumerate(axes[1][1:5]):
    if j < len(wrong_idx):
        idx = wrong_idx[j]
        ax.imshow(X_te_d[idx].reshape(8,8), cmap='gray')
        ax.set_title(f"True:{y_te_d[idx]} Pred:{y_pred_d[idx]}")
        ax.axis('off')

# T1303 – KNN Image Classification: Handwritten Digits
# Concepts: Image flattening, pixel features, KNN on visual data, misclassification analysis
plt.suptitle("Top: Sample Digits | Bottom-L: Confusion Matrix | Bottom-R: Misclassified")
plt.tight_layout(); plt.show()


In [ ]:
import time
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

wine = load_wine()
X_w  = StandardScaler().fit_transform(wine.data)
X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(X_w, wine.target,
                                                     test_size=0.2, stratify=wine.target, random_state=42)

algorithms = {'kd_tree': {}, 'brute': {}}
best_k_w = {}
for algo in algorithms:
    best_acc, best_k = 0, 1
    for k in range(1, 21):
        knn_w = KNeighborsClassifier(n_neighbors=k, algorithm=algo).fit(X_tr_w, y_tr_w)
        acc = accuracy_score(y_te_w, knn_w.predict(X_te_w))
        if acc > best_acc:
            best_acc, best_k = acc, k
    best_k_w[algo] = best_k

    knn_final = KNeighborsClassifier(n_neighbors=best_k_w[algo], algorithm=algo).fit(X_tr_w, y_tr_w)
    t0 = time.perf_counter()
    for _ in range(100): knn_final.predict(X_te_w)
    elapsed = (time.perf_counter() - t0) / 100 * 1000
    algorithms[algo] = {'best_k': best_k_w[algo], 'accuracy': best_acc, 'time_ms': elapsed}

print(f"{'Algorithm':<12} {'Best K':>7} {'Accuracy':>10} {'Inference (ms)':>16}")
print("-" * 50)
for algo, info in algorithms.items():
    print(f"{algo:<12} {info['best_k']:>7} {info['accuracy']:>10.4f} {info['time_ms']:>16.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
algos = list(algorithms.keys())
axes[0].bar(algos, [algorithms[a]['time_ms'] for a in algos], color=['steelblue','coral'])
axes[0].set_title('Inference Time (ms) Comparison'); axes[0].set_ylabel('ms')

# T1304 – KNN: KD-Tree vs Brute Force Efficiency Comparison
# Concepts: Spatial indexing, KD-tree, query time complexity, scalability
k_range = range(1, 21)
for algo in algos:
    accs = [accuracy_score(y_te_w, KNeighborsClassifier(n_neighbors=k, algorithm=algo)
                           .fit(X_tr_w, y_tr_w).predict(X_te_w)) for k in k_range]
    axes[1].plot(list(k_range), accs, label=algo, marker='o')
axes[1].set_title('K vs Accuracy'); axes[1].legend(); axes[1].set_xlabel('K')
plt.tight_layout(); plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

wine = load_wine()
X_w2  = StandardScaler().fit_transform(wine.data)
X_tr_w2, X_te_w2, y_tr_w2, y_te_w2 = train_test_split(X_w2, wine.target,
                                                         test_size=0.2, stratify=wine.target, random_state=42)
k_values = [1, 3, 5, 7, 9, 11, 13, 15]
cv_scores = {}
for k in k_values:
    scores = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_w2, wine.target, cv=5)
    cv_scores[k] = scores.mean()
    print(f"K={k:2d}  Mean CV Acc={scores.mean():.4f}  Std={scores.std():.4f}")

best_k_cv = max(cv_scores, key=cv_scores.get)
print(f"Best K = {best_k_cv}  (CV Acc = {cv_scores[best_k_cv]:.4f})")

final_knn = KNeighborsClassifier(n_neighbors=best_k_cv).fit(X_tr_w2, y_tr_w2)
y_pred_w2 = final_knn.predict(X_te_w2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(k_values, [cv_scores[k] for k in k_values], marker='o', color='steelblue')
axes[0].axvline(best_k_cv, color='red', linestyle='--', label=f'Best K={best_k_cv}')
axes[0].set_title('K vs Mean CV Accuracy'); axes[0].legend()

cm_w2 = confusion_matrix(y_te_w2, y_pred_w2)
sns.heatmap(cm_w2, annot=True, fmt='d', cmap='Purples', ax=axes[1])
axes[1].set_title('Confusion Matrix')

metrics = ['Precision','Recall','F1-Score']
vals = [precision_score(y_te_w2, y_pred_w2, average='macro'),
        recall_score(y_te_w2, y_pred_w2, average='macro'),
        f1_score(y_te_w2, y_pred_w2, average='macro')]
axes[2].bar(metrics, vals, color=['#4C72B0','#DD8452','#55A868'])
axes[2].set_title('Precision / Recall / F1'); axes[2].set_ylim(0, 1)
plt.tight_layout(); plt.show()

# T1305 – KNN Hyperparameter Tuning via 5-Fold Cross-Validation
# Concepts: Cross-validation, bias-variance trade-off, odd K values, generalisation
print("Analytical Answers:")
print("Q1. CV is more reliable: it uses all data for both training and validation, reducing variance.")
print("Q2. Odd K avoids ties in majority voting for binary classification.")
print(f"Q3. Best K = {best_k_cv}")
print("Q4. Low K (K=1) → high variance, memorises noise → overfitting.")
print("Q5. High K → over-smoothed boundary, underfits complex patterns.")


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data_cal2 = fetch_california_housing()
X_cal2, y_cal2 = data_cal2.data, data_cal2.target
X_tr_c2, X_te_c2, y_tr_c2, y_te_c2 = train_test_split(X_cal2, y_cal2, test_size=0.2, random_state=42)

# Depth experiment
depths_r, rmse_tr, rmse_te = [], [], []
for d in range(2, 21):
    dt_r = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_tr_c2, y_tr_c2)
    depths_r.append(d)
    rmse_tr.append(np.sqrt(mean_squared_error(y_tr_c2, dt_r.predict(X_tr_c2))))
    rmse_te.append(np.sqrt(mean_squared_error(y_te_c2, dt_r.predict(X_te_c2))))

best_depth_r = depths_r[np.argmin(rmse_te)]
best_dtr = DecisionTreeRegressor(max_depth=best_depth_r, random_state=42).fit(X_tr_c2, y_tr_c2)
y_pred_c2 = best_dtr.predict(X_te_c2)

print(f"Best depth: {best_depth_r}")
print(f"MAE  : {mean_absolute_error(y_te_c2, y_pred_c2):.4f}")
print(f"MSE  : {mean_squared_error(y_te_c2, y_pred_c2):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_te_c2, y_pred_c2)):.4f}")
print(f"R²   : {r2_score(y_te_c2, y_pred_c2):.4f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0][0].scatter(y_te_c2, y_pred_c2, alpha=0.3, color='steelblue', s=10)
axes[0][0].plot([y_te_c2.min(), y_te_c2.max()],[y_te_c2.min(), y_te_c2.max()],'r--')
axes[0][0].set_title('Actual vs Predicted')

axes[0][1].plot(depths_r, rmse_tr, label='Train RMSE', marker='o')
axes[0][1].plot(depths_r, rmse_te, label='Test RMSE',  marker='s')
axes[0][1].axvline(best_depth_r, color='red', linestyle='--', label=f'Best depth={best_depth_r}')
axes[0][1].set_title('Depth vs RMSE'); axes[0][1].legend()

fi_r = pd.Series(best_dtr.feature_importances_, index=data_cal2.feature_names).sort_values()
fi_r.plot.barh(ax=axes[1][0], color='teal'); axes[1][0].set_title('Feature Importance')

# T1306 – Decision Tree Regression: California Housing
# Concepts: Regression trees, complexity control, pruning, residual analysis
# Experiments with max_depth, min_samples_split, and ccp_alpha pruning. Reports MAE, RMSE, R², and plots depth vs RMSE and feature importance.
residuals = y_te_c2 - y_pred_c2
axes[1][1].hist(residuals, bins=40, color='salmon', edgecolor='black')
axes[1][1].set_title('Residual Distribution'); axes[1][1].set_xlabel('Residual')
plt.tight_layout(); plt.show()

## 14 K-Means & Clustering

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

# Synthetic Mall Customers data (replace with real CSV from Kaggle)
np.random.seed(42)
n_cust = 200
mall_df = pd.DataFrame({
    'CustomerID'    : range(1, n_cust+1),
    'Gender'        : np.random.choice(['Male','Female'], n_cust),
    'Age'           : np.random.randint(18, 70, n_cust),
    'Annual Income (k$)' : np.random.randint(15, 140, n_cust),
    'Spending Score (1-100)' : np.random.randint(1, 100, n_cust)
})
print(mall_df.head())
print(f"Shape: {mall_df.shape}")
print(f"Missing values:{mall_df.isnull().sum()}")

# L14 Setup – Load Mall Customers Dataset
# Concepts: Data loading, inspection, feature selection for distance-based clustering
features_num = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X_mall = mall_df[features_num].values
scaler_mall = StandardScaler()
X_mall_s = scaler_mall.fit_transform(X_mall)
print("Standardised feature matrix ready.")


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertia, sil_scores = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_mall_s)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_mall_s, km.labels_))

best_k14 = K_range.start + sil_scores.index(max(sil_scores))
print(f"Optimal K (best silhouette) = {best_k14}")

km_best = KMeans(n_clusters=best_k14, n_init=10, random_state=42).fit(X_mall_s)
mall_df['Cluster_KMeans'] = km_best.labels_

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(list(K_range), inertia, marker='o', color='steelblue')
axes[0].set_title('Elbow Curve'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')

axes[1].plot(list(K_range), sil_scores, marker='s', color='coral')
axes[1].axvline(best_k14, color='red', linestyle='--', label=f'Best K={best_k14}')
axes[1].set_title('Silhouette Scores'); axes[1].legend()

scatter = axes[2].scatter(X_mall_s[:,1], X_mall_s[:,2],
                          c=km_best.labels_, cmap='tab10', s=50, alpha=0.7)
axes[2].scatter(km_best.cluster_centers_[:,1], km_best.cluster_centers_[:,2],
                c='black', s=200, marker='X', label='Centroids')
axes[2].set_title('K-Means Clusters'); axes[2].set_xlabel('Income'); axes[2].set_ylabel('Spending')
axes[2].legend(); plt.tight_layout(); plt.show()

# T1401 – K-Means Customer Segmentation
# Concepts: K-Means, elbow method, silhouette score, centroid interpretation, cluster profiling
print("Cluster Behavioural Summary:")
print(mall_df.groupby('Cluster_KMeans')[features_num].mean().round(1).to_string())

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(X_mall_s, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
dendrogram(Z, truncate_mode='lastp', p=30, ax=axes[0], color_threshold=6)
axes[0].set_title('Dendrogram (Ward Linkage)'); axes[0].axhline(6, color='red', linestyle='--')

agg = AgglomerativeClustering(n_clusters=best_k14, linkage='ward').fit(X_mall_s)
mall_df['Cluster_Agg'] = agg.labels_

axes[1].scatter(X_mall_s[:,1], X_mall_s[:,2], c=agg.labels_, cmap='tab10', s=50, alpha=0.7)
axes[1].set_title('Agglomerative Clusters')
axes[1].set_xlabel('Income (scaled)'); axes[1].set_ylabel('Spending (scaled)')
plt.tight_layout(); plt.show()

# T1402 – Hierarchical (Agglomerative) Clustering
# Concepts: Agglomerative clustering, Ward linkage, dendrogram, merge distance analysis
# Comparison
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(mall_df['Cluster_KMeans'], mall_df['Cluster_Agg'])
print(f"Adjusted Rand Index (K-Means vs Agglomerative): {ari:.4f}")


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# One-hot encode Gender
gender_ohe = pd.get_dummies(mall_df['Gender'], prefix='Gender').values.astype(float)
X_mixed = np.hstack([X_mall_s, gender_ohe])

km_mixed = KMeans(n_clusters=best_k14, n_init=10, random_state=42).fit(X_mixed)
mall_df['Cluster_Mixed'] = km_mixed.labels_

sil_num   = silhouette_score(X_mall_s, mall_df['Cluster_KMeans'])
sil_mixed = silhouette_score(X_mixed,  mall_df['Cluster_Mixed'])

print(f"Silhouette (Numeric only) : {sil_num:.4f}")
print(f"Silhouette (With Gender)  : {sil_mixed:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Correlation heatmap
df_corr = mall_df[features_num + ['Cluster_KMeans']].copy()
sns.heatmap(df_corr.corr(), annot=True, cmap='coolwarm', ax=axes[0])
axes[0].set_title('Feature Correlation Heatmap')

axes[1].scatter(X_mall_s[:,1], X_mall_s[:,2], c=mall_df['Cluster_KMeans'],
                cmap='tab10', s=50); axes[1].set_title('Clusters (Numeric Only)')

axes[2].scatter(X_mall_s[:,1], X_mall_s[:,2], c=mall_df['Cluster_Mixed'],
                cmap='tab10', s=50); axes[2].set_title('Clusters (With Gender)')

# T1403 – Mixed-Type Clustering: Encoding Categorical Feature (Gender)
# Concepts: One-hot encoding, feature concatenation, comparing clustering with/without categoricals
plt.tight_layout(); plt.show()
bars = axes[0].figure.add_axes([0.35, -0.15, 0.3, 0.1])
bars.bar(['Numeric','With Gender'], [sil_num, sil_mixed], color=['steelblue','coral'])
bars.set_title('Silhouette Score Comparison'); bars.set_ylim(0, 0.5)
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

X_raw_mall = mall_df[features_num].values   # unscaled
X_scl_mall = X_mall_s                        # standardised

results_scale = {}
for label, X in [('Raw', X_raw_mall), ('Scaled', X_scl_mall)]:
    km = KMeans(n_clusters=best_k14, n_init=10, random_state=42).fit(X)
    agg = AgglomerativeClustering(n_clusters=best_k14, linkage='ward').fit(X)
    results_scale[label] = {
        'km_labels': km.labels_, 'agg_labels': agg.labels_,
        'inertia': km.inertia_,
        'sil_km':  silhouette_score(X, km.labels_),
        'sil_agg': silhouette_score(X, agg.labels_)
    }

print(f"{'Variant':<10} {'KMeans Inertia':>16} {'KMeans Sil':>12} {'Agg Sil':>10}")
print("-" * 52)
for lbl, res in results_scale.items():
    print(f"{lbl:<10} {res['inertia']:>16.2f} {res['sil_km']:>12.4f} {res['sil_agg']:>10.4f}")

# T1404 – Feature Scaling Impact on Distance-Based Clustering
# Concepts: Scale sensitivity, inertia comparison, silhouette score, normalization importance
# Empirically compares K-Means and Agglomerative clustering on raw (unscaled) versus standardised data.
# Shows how unscaled features with large magnitudes distort distance metrics.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (lbl, X) in enumerate([('Raw', X_raw_mall), ('Scaled', X_scl_mall)]):
    res = results_scale[lbl]
    axes[i][0].scatter(X[:,1], X[:,2], c=res['km_labels'], cmap='tab10', s=30, alpha=0.7)
    axes[i][0].set_title(f'K-Means ({lbl})')
    axes[i][1].scatter(X[:,1], X[:,2], c=res['agg_labels'], cmap='tab10', s=30, alpha=0.7)
    axes[i][1].set_title(f'Agglomerative ({lbl})')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering

X_out = X_mall_s.copy()
# Inject extreme outliers
outliers = np.array([[5, 5, 5], [-5, -5, -5], [6, -6, 6]])
X_with_out = np.vstack([X_out, outliers])

# IQR detection on original data
def iqr_mask(X):
    Q1, Q3 = np.percentile(X, 25, axis=0), np.percentile(X, 75, axis=0)
    IQR = Q3 - Q1
    return np.all((X >= Q1 - 1.5*IQR) & (X <= Q3 + 1.5*IQR), axis=1)

clean_mask = iqr_mask(X_with_out)
X_clean    = X_with_out[clean_mask]
print(f"Outliers detected/removed: {(~clean_mask).sum()}")

def run_km(X, k): return KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
km_with = run_km(X_with_out, best_k14)
km_no   = run_km(X_clean,    best_k14)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# Boxplots
for i, feat in enumerate(['Age','Income','Spending']):
    axes[0][0].boxplot([X_with_out[:,i], X_clean[:,i]], positions=[i*3+1, i*3+2])
axes[0][0].set_title('Boxplots: With vs Without Outliers')
axes[0][0].set_xticks([1,2,4,5,7,8]); axes[0][0].set_xticklabels(['Age+O','Age','Inc+O','Inc','Spe+O','Spe'])

axes[0][1].scatter(X_with_out[:,1], X_with_out[:,2], c=km_with.labels_, s=30, alpha=0.7, cmap='tab10')
axes[0][1].scatter(km_with.cluster_centers_[:,1], km_with.cluster_centers_[:,2],
                   c='black', s=200, marker='X'); axes[0][1].set_title('K-Means (With Outliers)')

axes[1][0].scatter(X_clean[:,1], X_clean[:,2], c=km_no.labels_, s=30, alpha=0.7, cmap='tab10')
axes[1][0].scatter(km_no.cluster_centers_[:,1], km_no.cluster_centers_[:,2],
                   c='black', s=200, marker='X'); axes[1][0].set_title('K-Means (Without Outliers)')

# T1405 – Outlier Impact on Clustering
# Concepts: IQR-based outlier detection, centroid shift, robust clustering analysis
# Runs K-Means and Hierarchical clustering before and after outlier removal, showing centroid displacement
# Centroid shift
shift = np.linalg.norm(km_with.cluster_centers_ - km_no.cluster_centers_, axis=1)
axes[1][1].bar(range(best_k14), shift, color='coral')
axes[1][1].set_title('Centroid Shift After Outlier Removal')
axes[1][1].set_xlabel('Cluster'); axes[1][1].set_ylabel('L2 Distance Shift')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage

iris2 = load_iris()
X_iris2 = StandardScaler().fit_transform(iris2.data)
y_iris2  = iris2.target

km_iris  = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_iris2)
agg_iris = AgglomerativeClustering(n_clusters=3, linkage='ward').fit(X_iris2)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# K-Means PCA
axes[0].scatter(X_pca[:,0], X_pca[:,1], c=km_iris.labels_, cmap='tab10', s=40)
axes[0].set_title('K-Means Clusters (PCA)')

# Agglomerative PCA
axes[1].scatter(X_pca[:,0], X_pca[:,1], c=agg_iris.labels_, cmap='tab10', s=40)
axes[1].set_title('Agglomerative Clusters (PCA)')

# Dendrogram
Z_iris = linkage(X_iris2, method='ward')
dendrogram(Z_iris, truncate_mode='lastp', p=15, ax=axes[2])
axes[2].set_title('Dendrogram')
plt.tight_layout(); plt.show()

# T1406 – Multi-Domain Clustering: Iris Dataset (Unsupervised Validation)
# Concepts: PCA dimensionality reduction, cluster-label mapping, ARI evaluation
# Confusion matrix (post-analysis only — labels not used during training)
print(f"KMeans  ARI vs true labels: {adjusted_rand_score(y_iris2, km_iris.labels_):.4f}")
print(f"Agglom  ARI vs true labels: {adjusted_rand_score(y_iris2, agg_iris.labels_):.4f}")
cm_iris2 = confusion_matrix(y_iris2, km_iris.labels_)
sns.heatmap(cm_iris2, annot=True, fmt='d', cmap='Blues',
            xticklabels=['C0','C1','C2'], yticklabels=iris2.target_names)
plt.title('Confusion Matrix: True Labels vs K-Means Clusters'); plt.tight_layout(); plt.show()

## 15

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class Perceptron:
    def __init__(self, lr=0.1, max_epochs=100):
        self.lr = lr; self.max_epochs = max_epochs

    def step(self, z): return 1 if z >= 0 else 0

    def fit(self, X, y):
        np.random.seed(42)
        self.w = np.random.randn(X.shape[1]) * 0.01
        self.b = 0.0
        self.epochs_run = 0
        for epoch in range(self.max_epochs):
            errors = 0
            for xi, yi in zip(X, y):
                y_hat = self.step(np.dot(xi, self.w) + self.b)
                err   = yi - y_hat
                self.w += self.lr * err * xi
                self.b += self.lr * err
                errors += int(err != 0)
            self.epochs_run = epoch + 1
            if errors == 0: break

    def predict(self, X):
        return np.array([self.step(np.dot(xi, self.w) + self.b) for xi in X])

# Datasets
X_or  = np.array([[0,0],[0,1],[1,0],[1,1]])
y_or  = np.array([0, 1, 1, 1])
X_and = np.array([[0,0],[0,1],[1,0],[1,1]])
y_and = np.array([0, 0, 0, 1])

print("OR Gate:")
for lr in [0.1, 0.01, 0.5]:
    p = Perceptron(lr=lr); p.fit(X_or, y_or)
    acc = np.mean(p.predict(X_or) == y_or)
    print(f"  LR={lr:4}  Epochs to converge: {p.epochs_run:3d}  Acc: {acc:.2f}")

print("AND Gate:")
for lr in [0.1, 0.01, 0.5]:
    p = Perceptron(lr=lr); p.fit(X_and, y_and)
    acc = np.mean(p.predict(X_and) == y_and)
    print(f"  LR={lr:4}  Epochs to converge: {p.epochs_run:3d}  Acc: {acc:.2f}")

# T1501 – Single-Layer Perceptron from Scratch (OR and AND Logic)
# Concepts: Perceptron learning rule, step activation, linear separability, convergence
# Decision boundary plot (OR, lr=0.1)
p_plot = Perceptron(lr=0.1); p_plot.fit(X_or, y_or)
plt.figure(figsize=(5,4))
plt.scatter(X_or[:,0], X_or[:,1], c=y_or, cmap='bwr', s=200, zorder=3)
x_line = np.linspace(-0.5, 1.5, 100)
if p_plot.w[1] != 0:
    y_line = -(p_plot.w[0]*x_line + p_plot.b) / p_plot.w[1]
    plt.plot(x_line, y_line, 'k--', label='Decision Boundary')
plt.title('Perceptron Decision Boundary (OR Gate)'); plt.legend()
plt.xlim(-0.5, 1.5); plt.ylim(-0.5, 1.5); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
X_act, y_act = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                    n_clusters_per_class=1, random_state=42)

def step_act(z):    return (z >= 0).astype(float)
def sigmoid_act(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def relu_act(z):    return np.maximum(0, z)

def perceptron_boundary(X, y, activation, epochs=500, lr=0.01):
    np.random.seed(42)
    X_b = np.hstack([np.ones((len(X),1)), X])
    w   = np.random.randn(X_b.shape[1]) * 0.01
    for _ in range(epochs):
        z = X_b @ w
        h = activation(z)
        w -= lr * (X_b.T @ (h - y)) / len(y)
    return w

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
xx, yy = np.meshgrid(np.linspace(X_act[:,0].min()-1, X_act[:,0].max()+1, 200),
                     np.linspace(X_act[:,1].min()-1, X_act[:,1].max()+1, 200))
grid = np.c_[np.ones(xx.ravel().shape), xx.ravel(), yy.ravel()]

# T1502 – Activation Function Comparison: Step, Sigmoid, ReLU
# Concepts: Activation functions, output surfaces, decision boundary geometry, bias-variance
# Compares Step, Sigmoid, and ReLU activation functions on a 2D Gaussian dataset.
# Plots decision boundaries for each and the sigmoid probability surface
for ax, (name, act) in zip(axes, [('Step', step_act), ('Sigmoid', sigmoid_act), ('ReLU', relu_act)]):
    w = perceptron_boundary(X_act, y_act, act)
    Z = act(grid @ w).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm', levels=20)
    ax.scatter(X_act[:,0], X_act[:,1], c=y_act, cmap='bwr', edgecolors='k', s=30)
    ax.set_title(f'{name} Activation — Decision Boundary')
plt.tight_layout(); plt.show()
print("Sigmoid produces smooth probability gradients; Step gives hard binary split; ReLU is asymmetric.")


In [ ]:
import numpy as np

X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([[0],[1],[1],[0]], dtype=float)

def sigmoid(z): return 1 / (1 + np.exp(-z))
def sigmoid_d(z): return z * (1 - z)

np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros((1, 1))
lr_xor, epochs_xor = 0.5, 10000

for epoch in range(1, epochs_xor + 1):
    # Forward
    z1  = X_xor @ W1 + b1;  a1 = sigmoid(z1)
    z2  = a1 @ W2 + b2;     a2 = sigmoid(z2)
    mse = np.mean((a2 - y_xor) ** 2)
    # Backprop
    d2  = (a2 - y_xor) * sigmoid_d(a2)
    d1  = (d2 @ W2.T)  * sigmoid_d(a1)
    W2 -= lr_xor * a1.T @ d2;  b2 -= lr_xor * d2.sum(axis=0, keepdims=True)
    W1 -= lr_xor * X_xor.T @ d1; b1 -= lr_xor * d1.sum(axis=0, keepdims=True)
    if epoch % 1000 == 0: print(f"Epoch {epoch:5d}  MSE = {mse:.6f}")

# T1503 – MLP from Scratch: XOR Problem
# Concepts: Backpropagation, hidden layer, non-linear separability, MSE convergence
# Builds a 2-4-1 MLP entirely with NumPy to solve the non-linearly separable XOR problem.
# Uses sigmoid activation and backpropagation; reports MSE every 1000 epochs.
print("Final Predictions:")
for xi, yi, pred in zip(X_xor, y_xor.flatten(), a2.flatten()):
    print(f"  Input: {xi}  Target: {yi}  Predicted: {pred:.4f}  Rounded: {round(pred)}")


In [ ]:
import numpy as np

def sigmoid(z): return 1 / (1 + np.exp(-z))

# Architecture: 2 inputs → 2 hidden → 1 output
np.random.seed(1)
W1 = np.random.randn(2, 2) * 0.5   # shape (2,2)
b1 = np.zeros(2)
W2 = np.random.randn(2, 1) * 0.5   # shape (2,1)
b2 = np.zeros(1)
lr_bp = 0.5

X_bp = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_bp = np.array([[0],[1],[1],[0]], dtype=float)  # XOR

# --- MANUAL single iteration on first sample ---
xi, yi = X_bp[0:1], y_bp[0:1]

# Forward
z1 = xi @ W1 + b1;  a1 = sigmoid(z1)
z2 = a1 @ W2 + b2;  a2 = sigmoid(z2)
error = 0.5 * (a2 - yi) ** 2

print("=== FORWARD PASS (Sample: [0,0] → 0) ===")
print(f"z1 = {z1}"); print(f"a1 = {a1}")
print(f"z2 = {z2}"); print(f"a2 = {a2}")
print(f"Error (MSE) = {error[0][0]:.6f}")

# Backward
d_out = (a2 - yi) * a2 * (1 - a2)          # output delta
d_hid = (d_out @ W2.T) * a1 * (1 - a1)    # hidden delta

print("
=== BACKWARD PASS ===")
print(f"Output delta : {d_out}")
print(f"Hidden delta : {d_hid}")
print(f"
Weight updates:")
print(f"  ΔW2 = {(-lr_bp * a1.T @ d_out)}")
print(f"  ΔW1 = {(-lr_bp * xi.T  @ d_hid)}")

# Update
W2 -= lr_bp * a1.T @ d_out
W1 -= lr_bp * xi.T  @ d_hid

print(f"Updated W1:{W1}")
print(f"Updated W2:{W2}")
# T1504 – Backpropagation Computational Simulation (2-2-1 MLP)
# Concepts: Forward pass, error computation, gradient backpropagation, weight update verification
# Manually simulates one full forward and backward pass of a 2-2-1 MLP, computing deltas and weight updates.
# Designed to verify the mathematical correctness of the backpropagation algorithm.
print("--- Verifying on all XOR samples after 1 update ---")
a1_all = sigmoid(X_bp @ W1 + b1)
a2_all = sigmoid(a1_all @ W2 + b2)
for row, pred in zip(X_bp, a2_all.flatten()):
    print(f"  {row} → {pred:.4f}")


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

bc3 = load_breast_cancer()
X_bc3, y_bc3 = bc3.data, bc3.target

scaler_bc3 = StandardScaler()
X_bc3_s = scaler_bc3.fit_transform(X_bc3)
X_tr3, X_te3, y_tr3, y_te3 = train_test_split(X_bc3_s, y_bc3, test_size=0.2, random_state=42)

mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
mlp.fit(X_tr3, y_tr3)
y_pred3 = mlp.predict(X_te3)

print("=== Classification Report ===")
print(classification_report(y_te3, y_pred3, target_names=['Malignant','Benign']))

# 5-fold cross-validation
cv_scores3 = cross_val_score(mlp, X_bc3_s, y_bc3, cv=5)
print(f"5-Fold CV Accuracies: {cv_scores3.round(4)}")
print(f"Mean CV Accuracy    : {cv_scores3.mean():.4f}")

# T1505 – MLPClassifier on Breast Cancer (Scikit-learn)
# Concepts: Multi-layer perceptron, sklearn API, cross-validation, classification metrics
# Trains an MLPClassifier on the Breast Cancer dataset using two hidden layers. 
# Reports confusion matrix, precision/recall/F1, and 5-fold cross-validation accuracy.
cm3 = confusion_matrix(y_te3, y_pred3)
sns.heatmap(cm3, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant','Benign'], yticklabels=['Malignant','Benign'])
plt.title('MLP Confusion Matrix – Breast Cancer'); plt.tight_layout(); plt.show()